### Validate that the new dataset is compatible with the old dataset
* Ensure that both datasets have the same columns
* Make several plots of all of the variables

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import uproot
from datetime import datetime
import plotly.graph_objects as go
import re
import awkward as ak
import pandas as pd

In [2]:
# new: https://github.com/cms-lpc-llp/run3_llp_analyzer/blob/main/lists/MDSNano/v2/MC_Sumto4B_MH-125-MS-15-ctauS-1000_TuneCP5_13p6TeV_powheg-pythia8.txt
# old: https://gitlab.nrp-nautilus.io/aaportel/mds-ml/-/blob/main/data/data-paths/ggH_dirs.txt?ref_type=heads#L100-L158


NEW_DATA_FILE = '/uscms/home/tlee/nobackup/work/run3_datagen/data/analyzer_output_mdsnano.root' 
NEW_RAW_DATA_FILE = '/uscms/home/tlee/nobackup/work/run3_datagen/data/samples/new-raw/PAT_NANO_1.root' 
OLD_DATA_FILE = '/uscms/home/tlee/nobackup/work/run3_datagen/data/samples/old/displacedJetMuon_ntupler_1.root' 

OUTPUT_DIR = '/uscms/home/tlee/nobackup/work/run3_datagen/notebooks/model_compatibility'
TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

old_root_file = uproot.open(OLD_DATA_FILE)
old_tree = old_root_file['ntuples']['llp']

new_root_file = uproot.open(NEW_DATA_FILE)
new_tree = new_root_file['MuonSystem']

new_raw_root_file = uproot.open(NEW_RAW_DATA_FILE)
new_raw_tree = new_raw_root_file['Events']

old_tree_keys = old_tree.keys()
new_tree_keys = new_tree.keys()
new_raw_tree_keys = new_raw_tree.keys()

# Pad the shorter lists to the same length
max_len = max(len(old_tree_keys), len(new_tree_keys), len(new_raw_tree_keys))
old_tree_keys += [''] * (max_len - len(old_tree_keys))
new_tree_keys += [''] * (max_len - len(new_tree_keys))
new_raw_tree_keys += [''] * (max_len - len(new_raw_tree_keys))


df = pd.DataFrame({
    'Old Tree (llp)': old_tree_keys,
    'New Raw Tree (Events)': new_raw_tree_keys,
    'New Tree (MuonSystem)': new_tree_keys,
})

df.head(df.shape[0])


,Old Tree (llp),New Raw Tree (Events),New Tree (MuonSystem)
0,isData,run,runNum
1,nPV,luminosityBlock,MC_condition
2,runNum,event,lumiSec
3,lumiNum,bunchCrossing,evtNum
4,eventNum,HTXS_njets25,mH
...,...,...,...
1938,,HLT_ExpressMuons,
1939,,HLT_OnlineMonitorGroup,
1940,,HLT_PPSMaxTracksPerArm1,
1941,,HLT_PPSMaxTracksPerRP4,


In [3]:
# Helper function to normalize keys: lowercase and strip special characters
def normalize_keys(keys):
    return {re.sub(r'\W+', '', key.lower()): key for key in keys}

# Normalize the keys
old_keys_normalized = normalize_keys(old_tree_keys)
new_keys_normalized = normalize_keys(new_tree_keys)

# Find intersection and differences
common_keys = set(old_keys_normalized.keys()) & set(new_keys_normalized.keys())
only_in_old = set(old_keys_normalized.keys()) - set(new_keys_normalized.keys())
only_in_new = set(new_keys_normalized.keys()) - set(old_keys_normalized.keys())

# Prepare report content
report_lines = []

report_lines.append(f"Common keys ({len(common_keys)}):")
for key in sorted(common_keys):
    report_lines.append(f" - Old: {old_keys_normalized[key]} | New: {new_keys_normalized[key]}")

report_lines.append(f"\nKeys only in OLD tree ({len(only_in_old)}):")
for key in sorted(only_in_old):
    report_lines.append(f" - {old_keys_normalized[key]}")

report_lines.append(f"\nKeys only in NEW tree ({len(only_in_new)}):")
for key in sorted(only_in_new):
    report_lines.append(f" - {new_keys_normalized[key]}")

# Join all lines
report_text = '\n'.join(report_lines)


In [4]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, "model_compatibility.txt")
with open(output_path, 'w') as f:
    f.write(report_text)

print(f"\nReport saved to: {output_path}")


Report saved to: /uscms/home/tlee/nobackup/work/run3_datagen/notebooks/model_compatibility/model_compatibility.txt


In [5]:
useless_cluster_columns = {
        "cscRechitClusterGenMuonDeltaR",
        "cscRechitClusterCaloJetVeto", 
        "cscRechitClusterVertexR", 
        "cscRechitClusterVertexZ", 
        "cscRechitClusterVertexDis", 
        "cscRechitClusterVertexChi2", 
        "cscRechitClusterVertexN1", 
        "cscRechitClusterVertexN5", 
        "cscRechitClusterVertexN10", 
        "cscRechitClusterVertexN15", 
        "cscRechitClusterVertexN20", 
        "cscRechitClusterVertexN",
        "cscRechitCluster_match_gParticle_index",
        "cscRechitCluster_match_gParticle_minDeltaR",
        "cscRechitCluster_match_gParticle_id",
    }

useless_rechit_columns = {"cscRechitsE", "cscRechitsChannels"}

def isValidRechitColumn(column_name):
    if column_name.startswith("cscRechit") and "Cluster" not in column_name and column_name not in useless_rechit_columns:
        return True
    return False

def isClusterColumn(column_name):
    if column_name.startswith("cscRechit") and "Cluster" in column_name and column_name not in useless_cluster_columns:
        return True
    return False

only_in_old_rechit_columns = [
    old_keys_normalized[k] for k in only_in_old
    if isValidRechitColumn(old_keys_normalized[k])
]
only_in_old_cluster_columns = [
    old_keys_normalized[k] for k in only_in_old
    if isClusterColumn(old_keys_normalized[k])
]


In [6]:
print(f"{len(only_in_old_rechit_columns) + len(only_in_old_cluster_columns)} variables left to implement")

for column in only_in_old_rechit_columns:
    print(column)

for column in only_in_old_cluster_columns:
    print(column)



17 variables left to implement
cscRechitsQuality
cscRechitsTpeak
cscRechitsNStrips
cscRechitsTwire
cscRechitsHitWire
cscRechitsChamber
cscRechitsWGroupsBX
cscRechitsNWireGroups
cscRechitsDetId
cscRechitsStation
cscRechitClusterMe11Ratio
cscRechitClusterTimeTotal
cscRechitClusterTimeSpreadWeighted
cscRechitClusterMe12Ratio
cscRechitCluster_match_cscSegCluster_index
cscRechitClusterNStation
cscRechitCluster_match_cscSegCluster_minDeltaR
